## 4、课后阅读：Static runtime Context

对应《尚硅谷-09-上下文与记忆.pdf》PDF 第 60-80 页的第 4 节。

按照课件顺序整理：**Node-style hooks → wrap_model_call → wrap_tool_call → dynamic_prompt → 工具中访问 Context**。
保留课件中的 `UserContext`、`CheckCredit`、`ToolGuard`、`UserInfo`、`read_user_info`、`write_user_info` 等命名与示例数据。

**运行方法**：选择项目的 `langchain` Python 内核，先运行“公共准备”和“聊天模型准备”，再按小节练习。
聊天模型沿用同目录笔记的百炼配置；`agent.invoke(...)` 会发送真实模型请求。
`InMemoryStore`、工具 schema 转换可离线观察，不需要数据库。

各组示例重新创建 `store` 和 `agent`；单独重复调用会继续使用已有记忆或额度，重新运行初始化会重置相应示例。
天气和新闻工具返回课件中的固定模拟文本，不代表实时天气或新闻。
文中的“预期现象”是观察指引，没有预填真实模型的执行结果。


### 4.0 先区分 State、Context 和 Store

静态运行时上下文（Static runtime Context）用于传入当前运行的用户身份、权限或连接等依赖。
在调用 `invoke / stream` 时通过 `context=...` 提供，框架在中间件和工具中通过运行时对象传递它。

| 对比 | State | Context | Store |
| --- | --- | --- | --- |
| 本章用途 | 消息列表、运行状态 | 本次调用的用户身份 | 用户额度、偏好、资料 |
| 常见入口 | `state["messages"]` | `runtime.context` | `runtime.store` |
| 创建 Agent 时配置 | `state_schema=...` | `context_schema=...` | `store=...` |
| 本例是否通过 Checkpointer 保存 | 工具综合示例会保存会话 State | 不作为会话状态自动保存；每次调用显式传入 | 独立于 Checkpointer 保存资料 |

课件第 60 页的文字把 `ContextSchema` 写成通过 `state_schema` 传入；以其后代码及当前接口为准，这里使用 **`context_schema=UserContext`**。
“静态”表示作为运行依赖传入、在本次执行中按不变数据使用，不表示 Python 的普通 `@dataclass` 本身禁止赋值。
`InMemoryStore` 的数据仍在内存中，重启内核后不会自动保留。


### 公共准备

统一导入后面各小节需要的类型和接口。工具中的 `ToolRuntime` 使用与相邻笔记相同的公开导入路径。
Node-style 接收 `Runtime`；Wrap-style 从 `request.runtime` 访问；工具函数接收自动注入的 `ToolRuntime`。


In [ ]:
import os
from dataclasses import dataclass
from pprint import pprint
from typing import Any, Callable, List

from dotenv import load_dotenv
from loguru import logger
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    ModelRequest,
    ModelResponse,
    ToolCallRequest,
    dynamic_prompt,
    hook_config,
)
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.utils.function_calling import convert_to_openai_tool

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command


### 聊天模型准备

沿用前面 Notebook 的百炼环境变量和模型名。此单元初始化客户端，后续的 `agent.invoke(...)` 才发送聊天请求。
只练习后面的 schema 转换时，不需要运行本单元。


In [ ]:
load_dotenv(override = True)

model = init_chat_model(
    model = "openai:qwen3.8-max",
    api_key = os.getenv("BAILIAN_API_KEY"),
    base_url = os.getenv("BAILIAN_API_URL"),
)


### 4.1 中间件中访问

中间件根据 `context` 中的用户身份定位 Store 数据，再决定是否继续调用、提供哪些工具，或者生成什么系统提示词。

#### 4.1.1 Node-style hooks

本例从长期记忆读取用户额度：未注册或剩余额度小于 3000 时，在 `before_model` 提前结束；正常调用后根据返回的 token 用量更新额度。

**① 准备用户额度与上下文类型。**


In [ ]:
store = InMemoryStore()
namespace = ("users", "credits")

key1 = "Ada Lovelace"
value1 = {
    "tokens_credit_left": 5000,
    "user_level": 5,
}
key2 = "Blackwell"
value2 = {
    "tokens_credit_left": 2999,
    "user_level": 5,
}
key3 = "Ampere"
value3 = {
    "tokens_credit_left": 1000,
    "user_level": 5,
}

store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)


@dataclass
class UserContext:
    username: str


**② 定义额度检查中间件。**

- `before_model`：通过 `runtime.context.username` 找到当前用户，决定继续或 `jump_to="end"`。
- `after_model`：读取最后一条模型消息的 `usage_metadata`，按课件公式扣减演示额度并写回 Store。
- `after_agent`：打印已注册用户的剩余额度；未注册用户没有记录，不再读取其 `value`。

`@hook_config(can_jump_to=["end"])` 声明该 hook 可以提前结束流程。
这里的 `input_tokens + output_tokens * 6` 是课件的教学规则，不是百炼服务的实际计费标准。


In [ ]:
class CheckCredit(AgentMiddleware):
    @hook_config(can_jump_to=["end"])
    def before_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        store = runtime.store
        context = runtime.context
        username = context.username
        credit = store.get(namespace, username)

        if not credit:
            return {
                "jump_to": "end",
                "messages": [AIMessage(content="您尚未注册~")],
            }
        if credit.value["tokens_credit_left"] < 3000:
            return {
                "jump_to": "end",
                "messages": [AIMessage(content=f"{username}额度不足，请充值")],
            }
        return None

    def after_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        store = runtime.store
        username = runtime.context.username
        credit = store.get(namespace, username)
        usage_metadata = state["messages"][-1].usage_metadata
        token_usage = (
            usage_metadata["input_tokens"] + usage_metadata["output_tokens"] * 6
        )
        credit.value["tokens_credit_left"] -= token_usage
        store.put(namespace, username, credit.value)
        return None

    def after_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        store = runtime.store
        username = runtime.context.username
        item = store.get(namespace, username)
        if item is not None:
            credit = item.value["tokens_credit_left"]
            logger.info(f"{username}当前剩余额度：{credit}")
        return None


In [ ]:
agent = create_agent(
    model = model,
    middleware = [CheckCredit()],
    store = store,
    context_schema = UserContext,
)


In [ ]:
response1 = agent.invoke(
    {"messages": ["你好啊，你知道 Ada Lovelace 的贡献吗？"]},
    context = UserContext(username="Ada Lovelace"),
)
print("=" * 30, "-> Ada Lovelace messages <-", "=" * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["你好啊，你知道 Blackwell 的贡献吗？"]},
    context = UserContext(username="Blackwell"),
)
print("=" * 30, "-> Blackwell messages <-", "=" * 30)
for msg in response2["messages"]:
    msg.pretty_print()


**预期观察**

| 用户 | 初始额度 | 预期路径 |
| --- | ---: | --- |
| Ada Lovelace | 5000 | 调用模型，按实际返回用量扣减 |
| Blackwell | 2999 | 提前返回“额度不足”，不调用模型 |
| Ampere | 1000 | 若以该用户调用，也会提前返回“额度不足” |
| Store 中不存在的用户 | 无记录 | 提前返回“您尚未注册” |

正常调用依赖模型返回 `usage_metadata`；没有该字段时无法按此示例计算，不能把缺失用量当作 0。
反复只运行调用单元会继续扣减，实际剩余额度以返回的用量为准，不使用 PDF 中的固定结果作为答案。


#### 4.1.2 Wrap-style hooks

##### ① wrap_model_call：根据用户身份筛选工具

本例通过 `request.runtime.context` 获取用户名，通过 `request.runtime.store` 查询其工具权限。
`request.override(tools=tools)` 只调整本次模型请求的工具集，随后调用 `handler(request)` 继续执行。

以下重新定义 `UserContext` 和 `CheckCredit`，覆盖上一小节同名定义；本小节的 `CheckCredit` 负责工具筛选。


In [ ]:
store = InMemoryStore()
namespace = ("users", "credits")

key1 = "Ada Lovelace"
value1 = {
    "tokens_left": 3000,
    "get_weather": "yes",
    "get_news": "no",
}
key2 = "Blackwell"
value2 = {
    "tokens_left": 2000,
    "get_weather": "no",
    "get_news": "yes",
}
key3 = "Ampere"
value3 = {
    "tokens_left": 3000,
    "get_weather": "yes",
    "get_news": "yes",
}

store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)


@tool
def get_weather(city: str) -> str:
    """查询指定城市当日天气（课件固定模拟数据）。"""
    return f"{city}今天天气不错"


@tool
def get_news() -> str:
    """查询当日新闻（课件固定模拟文本，非实时新闻）。"""
    return "美伊尚未达成停战协议"


@dataclass
class UserContext:
    username: str


class CheckCredit(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        context = request.runtime.context
        store = request.runtime.store
        value = store.get(namespace, context.username).value
        tools = []
        for tool in request.tools:
            tool_name = tool.name
            if value[tool_name] == "yes":
                tools.append(tool)
            else:
                logger.warning(f"{context.username}无权调用 {tool_name}")

        request = request.override(tools=tools)
        return handler(request)


In [ ]:
agent = create_agent(
    model = model,
    middleware = [CheckCredit()],
    tools = [get_weather, get_news],
    store = store,
    context_schema = UserContext,
)


In [ ]:
response1 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context = UserContext(username="Ada Lovelace"),
)
print("=" * 30, "-> Ada Lovelace messages <-", "=" * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context = UserContext(username="Blackwell"),
)
print("=" * 30, "-> Blackwell messages <-", "=" * 30)
for msg in response2["messages"]:
    msg.pretty_print()

response3 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context = UserContext(username="Ampere"),
)
print("=" * 30, "-> Ampere messages <-", "=" * 30)
for msg in response3["messages"]:
    msg.pretty_print()


**预期观察**

| 用户 | 本次暴露给模型的工具 |
| --- | --- |
| Ada Lovelace | `get_weather` |
| Blackwell | `get_news` |
| Ampere | `get_weather`、`get_news` |

工具执行后会再次调用模型，所以同一用户的工具筛选日志可能出现多次。
本例只根据权限筛选工具，尚未使用 `tokens_left` 扣减额度；下一小节补充工具执行前的检查。


##### ② wrap_tool_call：校验工具额度并更新长期记忆

基于上一小节的 `get_weather`、`get_news`、`UserContext` 和 `CheckCredit`，增加 `ToolGuard`。
先运行上一小节的定义单元；下面重新初始化 Store，避免沿用前次演示已经扣减的额度。

执行路径：**读取当前用户 → 查工具消耗 → 比较剩余额度 → 不足时返回工具失败消息 / 充足时更新 Store 并执行工具**。


In [ ]:
CREDIT_MAP = {
    "get_weather": 10,
    "get_news": 20,
}


class ToolGuard(AgentMiddleware):
    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
    ) -> ToolMessage | Command[Any]:
        store = request.runtime.store
        username = request.runtime.context.username
        value = store.get(namespace, username).value
        tool_name = request.tool.name
        credits_cost = CREDIT_MAP[tool_name]
        credits_left = value["tokens_left"]

        if credits_left < credits_cost:
            logger.warning(f"{username}额度不足，{tool_name}调用失败！")
            return Command(
                update={
                    "messages": [
                        ToolMessage(
                            content="额度不足，无法调用工具，请充值~",
                            tool_call_id=request.runtime.tool_call_id,
                        )
                    ]
                }
            )

        credits_left -= credits_cost
        value["tokens_left"] = credits_left
        store.put(namespace, username, value)
        logger.info(
            f"{username}调用 {tool_name} 花费 {credits_cost}，剩余额度 {credits_left}"
        )
        logger.info(
            f"{username}状态已更新，当前状态 {store.get(namespace, username).value}"
        )
        return handler(request)


In [ ]:
store = InMemoryStore()
namespace = ("users", "credits")

key1 = "Ada Lovelace"
value1 = {
    "tokens_left": 3000,
    "get_weather": "yes",
    "get_news": "no",
}
key2 = "Blackwell"
value2 = {
    "tokens_left": 2000,
    "get_weather": "no",
    "get_news": "yes",
}
key3 = "Ampere"
value3 = {
    "tokens_left": 3000,
    "get_weather": "yes",
    "get_news": "yes",
}

store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)

agent = create_agent(
    model = model,
    middleware = [CheckCredit(), ToolGuard()],
    tools = [get_weather, get_news],
    store = store,
    context_schema = UserContext,
)


In [ ]:
response = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context = UserContext(username="Ada Lovelace"),
)
print("=" * 30, "-> Ada Lovelace messages <-", "=" * 30)
for msg in response["messages"]:
    msg.pretty_print()

print("Store 中的当前额度与权限：")
pprint(store.get(namespace, "Ada Lovelace").value)


**预期观察**

Ada Lovelace 仅有天气权限；若本轮调用一次天气工具，则额度从 3000 变为 2990。
实际扣减次数由模型发出的工具调用决定，不把 2990 当作所有运行的固定结果。

额度不足时不调用 `handler`，也不扣额度，而是返回带有本次 `tool_call_id` 的 `ToolMessage`。
这表示当前工具调用失败，不表示自动终止整个 Agent；后续行为由模型及运行图决定。

保留课件的“先扣减再执行工具”顺序：如果工具实际抛出异常，本例没有自动退回额度的逻辑。
这里演示运行时访问和 Store 更新，不实现真实计费。


#### 4.1.3 特殊的便捷中间件 dynamic_prompt

通过用户身份读取长期偏好，在每次模型调用前生成系统提示词。
它调整的是本次模型请求中的系统消息，不会把提示词追加到返回的 `messages` 状态中。

**① 准备偏好、Context 与动态提示词函数。**


In [ ]:
namespace = ("users", "preferences")
key1 = "Ada Lovelace"
value1 = {
    "chat_preferences": [
        "不喜欢啰嗦",
        "尽可能用最精简的文字解释清楚",
    ]
}
key2 = "Blackwell"
value2 = {
    "chat_preferences": [
        "喜欢长篇大论、引经据典",
        "不知道的东西要明确说'不知道'，不能胡编乱造",
    ]
}

store = InMemoryStore()
store.put(namespace, key1, value1)
store.put(namespace, key2, value2)


@dataclass
class UserContext:
    username: str


@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    username = request.runtime.context.username
    store = request.runtime.store
    preferences = store.get(namespace, username).value["chat_preferences"]
    preferences_text = "\n".join(preferences)
    custom_prompt = f"# 用户偏好\n{preferences_text}"
    logger.info(f"{username}自定义系统提示词\n{custom_prompt}")
    return custom_prompt


In [ ]:
agent = create_agent(
    model = model,
    middleware = [personalized_prompt],
    store = store,
    context_schema = UserContext,
)


In [ ]:
response1 = agent.invoke(
    {"messages": ["为什么花儿这样红？"]},
    context = UserContext(username="Ada Lovelace"),
)
print("=" * 30, "-> Ada Lovelace messages <-", "=" * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["为什么花儿这样红？"]},
    context = UserContext(username="Blackwell"),
)
print("=" * 30, "-> Blackwell messages <-", "=" * 30)
for msg in response2["messages"]:
    msg.pretty_print()


**预期观察**

两次输入相同，但日志中的系统提示词不同：Ada Lovelace 偏好简短，Blackwell 偏好详细并说明不确定性。
模型通常会受这些偏好影响，具体文字和长度仍以真实输出为准。

返回的 `messages` 中看不到这次动态生成的系统提示词，不能据此认为提示词没有生效。
把课件中的偏好连接单独写成 `preferences_text`，便于观察字符串如何形成，其余含义不变。


### 4.2 工具中访问

工具参数可以使用 Pydantic 模型。下面先观察嵌套参数的工具 schema，再使用 `ToolRuntime` 读写用户资料。

#### 4.2.1 观察 UserInfo 与工具 schema

`user_info` 是模型提供的业务参数；`runtime` 是框架在执行工具时注入的对象，不要求模型生成。
本例的 `save_user_info` 只返回 `"ok"`，用于观察 schema，尚未写入 Store。
只运行公共导入和下面的单元即可，不需要调用聊天模型。


In [ ]:
class UserInfo(BaseModel):
    username: str = Field(description="用户名")
    age: int = Field(description="年龄")
    hobbies: List[str] = Field(description="兴趣爱好")


@tool(parse_docstring=True)
def save_user_info(user_info: UserInfo, runtime: ToolRuntime) -> str:
    """保存用户信息。

    Args:
        user_info: 用户信息对象。
    """
    return "ok"


pprint(convert_to_openai_tool(save_user_info))


观察 `function.parameters.properties`：应包含 `user_info`，其内部有 `username`、`age`、`hobbies`，但不包含 `runtime`。
这里三个字段均必填；下一组示例重新定义 `UserInfo`，增加默认值以便分次补充资料。

#### 4.2.2 通过工具读写长期记忆

保留课件中的韩立示例：第一轮记录姓名与爱好，第二轮补充年龄和新爱好，换一个 `thread_id` 后再读取同一用户资料。
当前用户通过 `context=UserContext(user_id=...)` 传入，不由模型填写用户 ID。

**① 定义 Context、Store、资料模型和两个工具。**


In [ ]:
@dataclass
class UserContext:
    user_id: str


store = InMemoryStore()
namespace = ("users", "user_info")


class UserInfo(BaseModel):
    username: str = Field(description="用户名", default="unknown")
    age: int = Field(description="年龄", default=0)
    hobbies: List[str] = Field(description="兴趣爱好", default_factory=list)


@tool(parse_docstring=True)
def read_user_info(runtime: ToolRuntime[UserContext, Any]) -> UserInfo | str:
    """读取用户信息。

    Returns:
        如果找到用户信息，返回 UserInfo 对象；否则返回空字符串。
    """
    user_id = runtime.context.user_id
    store = runtime.store
    item = store.get(namespace, user_id)
    if item:
        return UserInfo(**item.value)
    return ""


@tool(parse_docstring=True)
def write_user_info(
    user_info: UserInfo, runtime: ToolRuntime[UserContext, Any]
) -> bool:
    """将用户信息写入长期记忆。

    Args:
        user_info: 用户信息对象。

    Returns:
        如果成功写入则返回 True，否则返回 False。
    """
    user_id = runtime.context.user_id
    store = runtime.store
    try:
        logger.info(
            f"写入记忆，namespace: {namespace}, user_id: {user_id}, "
            f"user_info: {user_info.model_dump()}"
        )
        store.put(namespace, user_id, user_info.model_dump())
    except Exception as e:
        logger.error(e)
        return False
    return True


- `runtime.context.user_id`：找到本次用户；`runtime.store`：访问创建 Agent 时绑定的 Store。
- `UserInfo(**item.value)`：把字典解包为模型的字段；`model_dump()` 则把模型转回字典。
- `default_factory=list`：创建资料对象时为 `hobbies` 生成新的列表。
- `ToolRuntime[UserContext, Any]`：显式标注 Context 类型与 State 类型；`runtime` 仍由框架注入。

**② 创建 Agent 与两个会话配置。** 系统消息沿用课件，放在第一个会话的首次输入中。


In [ ]:
agent = create_agent(
    model = model,
    tools = [read_user_info, write_user_info],
    store = store,
    checkpointer = InMemorySaver(),
    context_schema = UserContext,
)

config1 = {"configurable": {"thread_id": "thread_1"}}
config2 = {"configurable": {"thread_id": "thread_2"}}


In [ ]:
response1 = agent.invoke(
    {
        "messages": [
            SystemMessage(
                "如果输入包含用户信息，抽取并调用工具记录。"
                "在记录之前先查找历史信息，和新增信息合并后记录。"
                "如果用户提问涉及个人信息，调用工具查找。"
            ),
            HumanMessage("你好，我是韩立，我喜欢修仙"),
        ]
    },
    context = UserContext(user_id="user_1"),
    config = config1,
)
for msg in response1["messages"]:
    msg.pretty_print()


**③ 在同一会话补充年龄与爱好。** 继续使用 `config1`，Checkpointer 会恢复 `thread_1` 的历史消息。


In [ ]:
thread_1_response = agent.invoke(
    {"messages": ["我今年两百岁了，是你们口中的“元婴老怪”，我喜欢跑步"]},
    context = UserContext(user_id="user_1"),
    config = config1,
)
print("=" * 30, "-> thread_1 messages <-", "=" * 30)
for msg in thread_1_response["messages"]:
    msg.pretty_print()


**④ 换一个会话，读取同一份用户资料。** `config2` 对应新的 `thread_2`，但 `user_id` 仍为 `user_1`。


In [ ]:
thread_2_response = agent.invoke(
    {"messages": ["你还记得我吗？"]},
    context = UserContext(user_id="user_1"),
    config = config2,
)
print("=" * 30, "-> thread_2 messages <-", "=" * 30)
for msg in thread_2_response["messages"]:
    msg.pretty_print()

print("直接检查长期记忆中的用户资料：")
pprint(store.get(namespace, "user_1"))


**预期流程与观察方法**

~~~text
thread_1，第 1 轮：我是韩立，我喜欢修仙
    → read_user_info 查旧资料
    → write_user_info 写入姓名与爱好

thread_1，第 2 轮：我今年两百岁了，我喜欢跑步
    → 读取历史资料，合并后写回
    → username=韩立，age=200，hobbies=[修仙, 跑步]

thread_2，第 1 轮：你还记得我吗？
    → 新会话历史中没有 thread_1 的对话
    → 用同一个 user_id 调用 read_user_info
    → 从同一个 Store 读取已保存资料
~~~

以上是课件要演示的工具路径，不是已执行日志。模型是否按顺序调用、是否正确合并，需要检查工具调用和 Store 的实际内容。
`put()` 替换该记录的字典；这里的资料合并依靠模型遵循提示词，并非 Store 自动完成。

`thread_2` 不会继承第一个会话输入中的 `SystemMessage`。本笔记保留课件写法；
如果所有会话都需要同一套规则，可在后续练习中把规则配置为 `create_agent(..., system_prompt=...)`。

### 本节回顾

1. `thread_id` 决定恢复哪份短期状态；`user_id` 与 namespace/key 决定读哪份用户资料。
2. `context` 是每次调用传入的依赖，使用相同会话也要显式传入，不依靠检查点自动恢复。
3. 不同会话能通过相同的 Store、namespace 和用户 key 访问同一条资料；换用户标识会访问不同资料。
4. `Runtime` 用于图和中间件；`ToolRuntime` 用于工具；它们都可以提供 Context 和 Store。
5. `runtime` 不在工具对模型暴露的参数 schema 中，工具运行时由框架注入。
6. `InMemorySaver` 与 `InMemoryStore` 分别保存会话状态和用户资料；重启内核后，本节内存数据都会丢失。

**来源与必要差异**：主体来自课件 PDF 第 60-80 页。使用相邻 Notebook 的百炼模型配置和公开 `ToolRuntime` 导入；
修正 `context_schema` 的文字说明，在额度示例的 `after_agent` 中避免读取不存在用户的记录，并把所有模型输出留待实际运行观察。
接口按项目现有 `langchain==1.2.12`、`langchain-core==1.2.18`、`langgraph==1.1.2`、Python 3.13.12 核对，未升级依赖。
